# 

In [ ]:
ssh-keygen -f "/home/khaled/.ssh/known_hosts" -R "172.16.6.66"

In [ ]:
sudo bash -c 'echo "PermitRootLogin yes" > /etc/ssh/sshd_config.d/01-permitroot.conf'
sudo systemctl restart sshd

In [ ]:
# For terminal 
systemctl set-default multi-user.target

# For GUI
systemctl set-default graphical.target

In [ ]:
hostnamectl hostname 

In [ ]:
sudo dnf install -y bash-completion

source /etc/profile.d/bash_completion.sh

In [ ]:
nmcli connection modify "ens192" ipv4.dns "172.16.6.70 172.16.1.16 172.16.1.20"

nmcli connection modify "ens192" ipv4.dns-search "voip.local alkancit.local"

nmcli connection modify "ens192" ipv4.ignore-auto-dns yes

nmcli connection down "ens192"
nmcli connection up "ens192"


cat /etc/resolv.conf

---

Storage steps

On stoarge VM

In [ ]:
# On the storage-vm (Rocky/RHEL 9)
dnf install -y targetcli

# Create the backing file from the second disk
targetcli

# Inside targetcli shell:
# /> cd backstores/block
/backstores/block> create xdr-lun /dev/sdb

# Create iSCSI target
# /> cd /iscsi
/iscsi> create iqn.2026-04.net.agency:storage-vm

# Create LUN
/iscsi/iqn.2026-04.net.agency:storage-vm/tpg1> luns/ create /backstores/block/xdr-lun

# Set ACLs — only your 3 workers can connect
/iscsi/iqn.2026-04.net.agency:storage-vm/tpg1> acls/ create iqn.2026-04.net.agency:worker-1
/iscsi/iqn.2026-04.net.agency:storage-vm/tpg1> acls/ create iqn.2026-04.net.agency:worker-2
/iscsi/iqn.2026-04.net.agency:storage-vm/tpg1> acls/ create iqn.2026-04.net.agency:worker-3

set attribute authentication=0

set attribute demo_mode_write_protect=0
set attribute generate_node_acls=1

set auth userid=iscsi-user password=1

# Save and exit

/> exit

systemctl enable --now target

In [ ]:
systemctl status target
targetcli ls

On all worker

In [ ]:
# For centos
sudo dnf config-manager --set-enabled highavailability


# Now install the packages
dnf install -y pacemaker pcs corosync fence-agents-all

systemctl enable --now pcsd iscsid multipathd



# Or if using EPEL
dnf install -y centos-release-nfs-ganesha5

dnf install -y nfs-ganesha nfs-ganesha-vfs

In [ ]:

# Set unique IQN per worker
# On worker-1:
echo "InitiatorName=iqn.2026-04.net.agency:worker-1" > /etc/iscsi/initiatorname.iscsi

# On worker-2:
echo "InitiatorName=iqn.2026-04.net.agency:worker-2" > /etc/iscsi/initiatorname.iscsi

# On worker-3:
echo "InitiatorName=iqn.2026-04.net.agency:worker-3" > /etc/iscsi/initiatorname.iscsi

In [ ]:
cat > /etc/iscsi/iscsid.conf <<'EOF'
node.startup = automatic
node.session.timeo.replacement_timeout = 15
node.conn[0].timeo.noop_out_interval = 5
node.conn[0].timeo.noop_out_timeout = 10
node.session.err_timeo.abort_timeout = 15
node.session.err_timeo.lu_reset_timeout = 20
node.session.cmds_max = 1024
node.session.queue_depth = 128
# node.session.auth.authmethod = CHAP
# node.session.auth.username = iscsi-user
# node.session.auth.password = 1
EOF

systemctl restart iscsid

In [ ]:
# sed -i 's/^node.session.auth.authmethod/#node.session.auth.authmethod/' /etc/iscsi/iscsid.conf
# sed -i 's/^node.session.auth.username/#node.session.auth.username/' /etc/iscsi/iscsid.conf
# sed -i 's/^node.session.auth.password/#node.session.auth.password/' /etc/iscsi/iscsid.conf

In [ ]:
# discovery
iscsiadm -m discovery -t sendtargets -p 172.16.6.60:3260
iscsiadm -m node --login
iscsiadm -m session  # should show 1 session (storage-vm has 1 IP, not dual-controller)

# iscsiadm -m node --logoutall=all
# iscsiadm -m node -o delete

Only on worker-1

In [ ]:
# # On worker-1 ONLY:
# mkfs.xfs -f /dev/sdb   # or whatever device the iSCSI LUN appears as
# mkdir -p /srv/xdr/pv-root
# mount /dev/sdb /srv/xdr/pv-root

In [ ]:
mkfs.xfs -f -L xdr-pv-root /dev/sdb
mkdir -p /srv/xdr/pv-root
mount -L xdr-pv-root /srv/xdr/pv-root
df -h /srv/xdr/pv-root
# Should show ~100G, xfs

In [ ]:
# mkdir -p /srv/xdr/pv-root/{clickhouse,logs,shared}
# chown -R 10001:10001 /srv/xdr/pv-root/clickhouse

---

On all Worker

In [ ]:
# Same password on ALL three workers
echo "1" | passwd --stdin hacluster

On worker-1

Pacemaker Cluster Setup

In [ ]:
pcs host auth worker-1 worker-2 worker-3 -u hacluster -p 1

pcs cluster setup storage-ha-cluster worker-1 worker-2 worker-3
pcs cluster start --all
pcs cluster enable --all

pcs status
# Should show: 3 nodes online, no resources yet

STONITH Configuration

In [ ]:
# Replace 10.0.0.5 with your ESXi host IP
# Replace "YourESXiRootPassword" with actual root password


pcs stonith create fence-w1 fence_vmware_soap \
    pcmk_host_list="worker-1" \
    ip="172.16.0.87" \
    username="root" \
    password="root@123" \
    ssl_insecure=1 \
    pcmk_reboot_action="reboot" \
    op monitor interval=60s

pcs stonith create fence-w2 fence_vmware_soap \
    pcmk_host_list="worker-2" \
    ip="172.16.0.87" \
    username="root" \
    password="root@123" \
    ssl_insecure=1 \
    pcmk_reboot_action="reboot" \
    op monitor interval=60s

pcs stonith create fence-w3 fence_vmware_soap \
    pcmk_host_list="worker-3" \
    ip="172.16.0.87" \
    username="root" \
    password="root@123" \
    ssl_insecure=1 \
    pcmk_reboot_action="reboot" \
    op monitor interval=60s

In [ ]:
# Remove the fence-w1 resource
# pcs stonith delete fence-w1

# Check STONITH resources
# pcs stonith show

# Or list all STONITH devices
# pcs stonith list


In [ ]:
pcs constraint location fence-w1 avoids worker-1
pcs constraint location fence-w2 avoids worker-2
pcs constraint location fence-w3 avoids worker-3

In [ ]:
 # clear the steps

NFS-Ganesha Configuration ( ALL Three Workers )

In [ ]:
cat > /etc/ganesha/ganesha.conf <<'EOF'
NFS_CORE_PARAM {
    NFS_Port = 2049;
    Protocols = 4;
}

EXPORT_DEFAULTS {
    Access_Type = None;
}

EXPORT {
    Export_Id = 1;
    Path = "/srv/xdr/pv-root";
    Pseudo = "/xdr";
    Protocols = 4;
    Transports = TCP;
    Access_Type = RW;
    Squash = No_Root_Squash;

    CLIENT {
        Clients = 172.16.0.0/16;
        Access_Type = RW;
        Squash = No_Root_Squash;
    }

    FSAL {
        Name = VFS;
    }
}

LOG {
    Default_Log_Level = WARN;
    Components {
        FSAL = INFO;
        NFS4 = WARN;
    }
}
EOF

Disable systemd Control (Pacemaker will manage it)

In [ ]:
# On ALL workers
systemctl disable nfs-ganesha
systemctl stop nfs-ganesha

Pacemaker Resources ( `Only worker-1` )

- Create Floating VIP

In [ ]:
pcs resource create storage-vip IPaddr2 \
    ip=172.16.6.113 \
    cidr_netmask=16 \
    nic=ens192 \
    op monitor interval=10s timeout=20s

- Create Filesystem Resource

In [ ]:
pcs resource create storage-lun Filesystem \
    device="/dev/disk/by-label/xdr-pv-root" \
    directory="/srv/xdr/pv-root" \
    fstype="xfs" \
    options="noatime,nodiratime,inode64" \
    op start timeout=90s \
    op stop timeout=90s \
    op monitor interval=20s timeout=40s

- Create NFS-Ganesha Resource

In [ ]:
pcs resource create nfs-ganesha systemd:nfs-ganesha \
    op start timeout=30s \
    op stop timeout=30s \
    op monitor interval=10s timeout=20s

- Group Resources (Order Matters)

In [ ]:
pcs resource group add storage-group \
    storage-vip storage-lun nfs-ganesha

- Prefer worker-1 as Active

In [ ]:
# Step 1: Set stickiness so resources don't bounce back
pcs resource defaults update  resource-stickiness=200
# Step 2: Keep lower preferences just as tiebreakers
pcs constraint location storage-group prefers \
    worker-1=50 worker-2=50 worker-3=50
# Equal preference = stays wherever it currently runs

In [ ]:
# pcs constraint location storage-group prefers worker-1=100 worker-2=50 worker-3=25

- Verify

In [ ]:
pcs status
pcs resource config show storage-group

Verify NFS Export (From Any Node (or master))

In [ ]:
# Test mount
mkdir -p /mnt/test
mount -t nfs4 172.16.6.113:/xdr /mnt/test
df -h /mnt/test
ls -la /mnt/test
touch /mnt/test/verify.txt
ls /mnt/test/
umount /mnt/test

Kubernetes Setup (Brief — Your Existing Process)

- Install NFS CSI Driver (on master-1)

In [ ]:
helm repo add csi-driver-nfs https://raw.githubusercontent.com/kubernetes-csi/csi-driver-nfs/master/charts
helm install csi-driver-nfs csi-driver-nfs/csi-driver-nfs \
    --namespace kube-system

- Create StorageClass

In [ ]:
vim storage-class-nfs.yaml

In [ ]:
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: shared-storage
provisioner: nfs.csi.k8s.io
parameters:
  server: "172.16.6.113"      # ← Pacemaker floating VIP
  share: "/xdr"
  subDir: "${pvc.metadata.namespace}/${pvc.metadata.name}"
mountOptions:
  - hard
  - nointr
  - nfsvers=4.1
  - rsize=1048576
  - wsize=1048576
  - timeo=600
  - retrans=2
  - noatime
  - nodiratime
reclaimPolicy: Retain
volumeBindingMode: Immediate

In [ ]:
kubectl apply -f storage-class-nfs.yaml

- ClickHouse PVC Example

In [ ]:
kubectl create namespace clickhouse

In [ ]:
vim clickhouse-pvc.yaml

In [ ]:
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: clickhouse-data
  namespace: clickhouse
spec:
  accessModes:
    - ReadWriteOnce
  storageClassName: shared-storage
  resources:
    requests:
      storage: 10Gi

In [ ]:
kubectl apply -f clickhouse-pvc.yaml

- Create ClickHouse StatefulSet

In [ ]:
cat > clickhouse.yaml <<'EOF'
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: clickhouse
  namespace: clickhouse
spec:
  serviceName: clickhouse
  replicas: 1
  selector:
    matchLabels:
      app: clickhouse
  template:
    metadata:
      labels:
        app: clickhouse
    spec:
      terminationGracePeriodSeconds: 30

      tolerations:
      - key: node.kubernetes.io/not-ready
        operator: Exists
        effect: NoExecute
        tolerationSeconds: 60
      - key: node.kubernetes.io/unreachable
        operator: Exists
        effect: NoExecute
        tolerationSeconds: 60

      affinity:                                          # ← ADD HERE
        nodeAffinity:                                    # inside spec
          preferredDuringSchedulingIgnoredDuringExecution:
          - weight: 100
            preference:
              matchExpressions:
              - key: storage-vip
                operator: In
                values:
                - "true"

      containers:
      - name: clickhouse
        image: clickhouse/clickhouse-server:24.3
        ports:
        - containerPort: 8123
          name: http
        - containerPort: 9000
          name: native
        volumeMounts:
        - name: data
          mountPath: /var/lib/clickhouse
        - name: logs
          mountPath: /var/log/clickhouse-server
        resources:
          requests:
            memory: "1Gi"
            cpu: "500m"

      volumes:
      - name: data
        persistentVolumeClaim:
          claimName: clickhouse-data
      - name: logs
        emptyDir: {}
---
apiVersion: v1
kind: Service
metadata:
  name: clickhouse
  namespace: clickhouse
spec:
  selector:
    app: clickhouse
  ports:
  - port: 8123
    name: http
  - port: 9000
    name: native
EOF
kubectl apply -f clickhouse.yaml

- creat PodDisruptionBudget

In [ ]:
vim pdb.yaml

In [ ]:
apiVersion: policy/v1
kind: PodDisruptionBudget
metadata:
  name: clickhouse-pdb
  namespace: clickhouse
spec:
  minAvailable: 1
  selector:
    matchLabels:
      app: clickhouse

In [ ]:
kubectl apply -f pdb.yaml

- Edit eviction time

In [ ]:
vim /etc/kubernetes/manifests/kube-controller-manager.yaml

In [ ]:
spec:
  containers:
  - command:
    - --node-monitor-grace-period=20s
    - --pod-eviction-timeout=30s

- Delete the pod if you need
  

In [ ]:
kubectl delete pod clickhouse-0 -n clickhouse --force --grace-period=0

- Create Test Database and Table with Data

In [ ]:
# Exec into ClickHouse pod
kubectl exec -it -n clickhouse clickhouse-0 -- clickhouse-client

In [ ]:
-- Create database
CREATE DATABASE IF NOT EXISTS testdb;

-- Create table with MergeTree engine (persists to disk)
CREATE TABLE testdb.failover_test (
    id UInt64,
    message String,
    created_at DateTime DEFAULT now()
) ENGINE = MergeTree()
ORDER BY id;

-- Insert test data
INSERT INTO testdb.failover_test (id, message) VALUES
(1, 'Data before failover - record 1'),
(2, 'Data before failover - record 2'),
(3, 'Data before failover - record 3');

-- Verify data exists
SELECT * FROM testdb.failover_test;

-- Exit
exit

Verify Data Written to NFS Backend

In [ ]:
# On worker-1 (where storage is currently active)
ls -la /srv/xdr/pv-root/clickhouse/clickhouse-data/
# Should see ClickHouse data files

# Or mount NFS and check from any node
mkdir -p /mnt/nfs-check
mount -t nfs4 172.16.6.113:/xdr /mnt/nfs-check
ls -la /mnt/nfs-check/clickhouse/clickhouse-data/
ls -la /mnt/nfs-check/clickhouse/clickhouse-data/data/testdb/
umount /mnt/nfs-check

Failover Test — Prove Data Survives

- Note Current State

In [ ]:
# Check where ClickHouse is running
kubectl get pod -n clickhouse clickhouse-0 -o wide
# Note the node

# Check where storage-group is active
pcs status
# Should show worker-1 (if you followed the preference)

---

Remove every thing

- On ALL workers: Clean up first

In [ ]:
pcs cluster stop
pcs cluster destroy
rm -rf /var/lib/pacemaker/* /var/lib/corosync/*
systemctl stop pacemaker corosync

---

# solution

In [ ]:
vim auto-force-delete-terminating.yaml

In [ ]:
kubectl apply -f - <<'EOF'
apiVersion: batch/v1
kind: CronJob
metadata:
  name: pod-reaper
  namespace: kube-system
spec:
  schedule: "*/1 * * * *"
  concurrencyPolicy: Forbid
  jobTemplate:
    spec:
      template:
        spec:
          serviceAccountName: pod-reaper
          restartPolicy: OnFailure
          tolerations:
          - key: node.kubernetes.io/not-ready
            operator: Exists
            effect: NoExecute
            tolerationSeconds: 30
          - key: node.kubernetes.io/unreachable
            operator: Exists
            effect: NoExecute
            tolerationSeconds: 30
          containers:
          - name: reaper
            image: bitnami/kubectl:latest
            command:
            - /bin/sh
            - -c
            - |
              echo "=== Pod Reaper Start ==="

              NOT_READY_NODES=$(kubectl get nodes --no-headers \
                | awk '$2=="NotReady" {print $1}')

              if [ -z "$NOT_READY_NODES" ]; then
                echo "All nodes Ready. Exiting."
                exit 0
              fi

              echo "NotReady nodes: $NOT_READY_NODES"

              for NODE in $NOT_READY_NODES; do
                echo "--- Processing node: $NODE ---"

                STUCK_PODS=$(kubectl get pods --all-namespaces \
                  --no-headers \
                  --field-selector="spec.nodeName=${NODE}" \
                  | awk '$4=="Terminating" {print $1" "$2}')

                if [ -z "$STUCK_PODS" ]; then
                  echo "No Terminating pods on $NODE"
                  continue
                fi

                echo "$STUCK_PODS" | while read NS NAME; do
                  echo "Force deleting: $NAME in $NS"
                  kubectl delete pod "$NAME" -n "$NS" --force --grace-period=0
                done
              done

              echo "=== Pod Reaper Done ==="
EOF

In [ ]:
kubectl get pods -n kube-system | grep reape

In [ ]:
kubectl apply -f auto-force-delete-terminating.yaml

---

In [ ]:
# ------------------------------

In [ ]:
vim /etc/kubernetes/manifests/kube-controller-manager.yaml 

In [ ]:
spec:
  containers:
  - command:
    - --node-monitor-grace-period=40s
    - --pod-eviction-timeout=60s

In [ ]:
# ------------------------------

In [ ]:
vim clickhouse.yaml 

In [ ]:
      tolerations:
      - key: node.kubernetes.io/not-ready
        operator: Exists
        effect: NoExecute
        tolerationSeconds: 60

      - key: node.kubernetes.io/unreachable
        operator: Exists
        effect: NoExecute
        tolerationSeconds: 60

---

In [ ]:
# Check cronjob status
kubectl get cronjob pod-reaper -n kube-system

# Check if any jobs were created
kubectl get jobs -n kube-system

# Check the job pods
kubectl get pods -n kube-system | grep reaper

kubectl logs -n kube-system pod-reaper-29673710-nl89r 

kubectl logs -n kube-system pod-reaper-29673708-bbb2q 


kubectl get nodes
kubectl describe node worker-1 | grep -A5 "Conditions:"

- On all worker nodes

In [ ]:
cat > /usr/local/bin/k8s-label-storage.sh <<'EOF'
#!/bin/bash
ACTION=$1
NODE=$(hostname)
KUBECONFIG=/etc/kubernetes/kubelet.conf

case "$ACTION" in
  start)
    echo "VIP starting on $NODE - setting label"
    kubectl --kubeconfig $KUBECONFIG \
      label node $NODE storage-vip=true --overwrite
    ;;
  stop)
    echo "VIP stopping on $NODE - removing label"
    kubectl --kubeconfig $KUBECONFIG \
      label node $NODE storage-vip=false --overwrite
    ;;
esac
exit 0
EOF

chmod +x /usr/local/bin/k8s-label-storage.sh

- On all worker nodes
 

In [ ]:
cat > /etc/systemd/system/k8s-vip-label.service <<'EOF'
[Unit]
Description=K8s node label for storage VIP
After=network.target

[Service]
Type=oneshot
ExecStart=/usr/local/bin/k8s-label-storage.sh start
ExecStop=/usr/local/bin/k8s-label-storage.sh stop
RemainAfterExit=yes

[Install]
WantedBy=multi-user.target
EOF

systemctl daemon-reload

systemctl enable k8s-vip-label   # Now works
systemctl disable k8s-vip-label  # Now works

- On `worker-1 Only` 

In [ ]:
pcs resource create k8s-vip-label systemd:k8s-vip-label \
  op start timeout=30s \
  op stop  timeout=30s \
  op monitor interval=60s timeout=30s

pcs resource group add storage-group k8s-vip-label

pcs status

# Ask claude

In [ ]:
[root@master-1 storage]# cat clickhouse.yaml 
apiVersion: apps/v1
kind: StatefulSet
metadata:
  name: clickhouse
  namespace: clickhouse
spec:
  serviceName: clickhouse
  replicas: 1
  selector:
    matchLabels:
      app: clickhouse
  template:
    metadata:
      labels:
        app: clickhouse
    spec:
      terminationGracePeriodSeconds: 30

      tolerations:
      - key: node.kubernetes.io/not-ready
        operator: Exists
        effect: NoExecute
        tolerationSeconds: 30

      - key: node.kubernetes.io/unreachable
        operator: Exists
        effect: NoExecute
        tolerationSeconds: 30

      containers:
      - name: clickhouse
        image: clickhouse/clickhouse-server:24.3
        ports:
        - containerPort: 8123
          name: http
        - containerPort: 9000
          name: native
        volumeMounts:
        - name: data
          mountPath: /var/lib/clickhouse
        - name: logs
          mountPath: /var/log/clickhouse-server
        resources:
          requests:
            memory: "1Gi"
            cpu: "500m"
      volumes:
      - name: data
        persistentVolumeClaim:
          claimName: clickhouse-data
      - name: logs
        emptyDir: {}
---
apiVersion: v1
kind: Service
metadata:
  name: clickhouse
  namespace: clickhouse
spec:
  selector:
    app: clickhouse
  ports:
  - port: 8123
    name: http
  - port: 9000
    name: native
[root@master-1 storage]# 

[root@master-1 storage]# cat /etc/kubernetes/manifests/kube-controller-manager.yaml
apiVersion: v1
kind: Pod
metadata:
  labels:
    component: kube-controller-manager
    tier: control-plane
  name: kube-controller-manager
  namespace: kube-system
spec:
  containers:
  - command:
    - --node-monitor-grace-period=20s
    - --pod-eviction-timeout=30s
    - kube-controller-manager
    - --allocate-node-cidrs=true
    - --authentication-kubeconfig=/etc/kubernetes/controller-manager.conf
    - --authorization-kubeconfig=/etc/kubernetes/controller-manager.conf
    - --bind-address=127.0.0.1
    - --client-ca-file=/etc/kubernetes/pki/ca.crt
    - --cluster-cidr=10.244.0.0/16
    - --cluster-name=kubernetes
    - --cluster-signing-cert-file=/etc/kubernetes/pki/ca.crt
    - --cluster-signing-key-file=/etc/kubernetes/pki/ca.key
    - --controllers=*,bootstrapsigner,tokencleaner
    - --kubeconfig=/etc/kubernetes/controller-manager.conf
    - --leader-elect=true
    - --requestheader-client-ca-file=/etc/kubernetes/pki/front-proxy-ca.crt
    - --root-ca-file=/etc/kubernetes/pki/ca.crt
    - --service-account-private-key-file=/etc/kubernetes/pki/sa.key
    - --service-cluster-ip-range=10.96.0.0/12
    - --use-service-account-credentials=true
    image: registry.k8s.io/kube-controller-manager:v1.35.5
    imagePullPolicy: IfNotPresent
    livenessProbe:
      failureThreshold: 8
      httpGet:
        host: 127.0.0.1
        path: /healthz
        port: probe-port
        scheme: HTTPS
      initialDelaySeconds: 10
      periodSeconds: 10
      timeoutSeconds: 15
    name: kube-controller-manager
    ports:
    - containerPort: 10257
      name: probe-port
      protocol: TCP
    resources:
      requests:
        cpu: 200m
    startupProbe:
      failureThreshold: 24
      httpGet:
        host: 127.0.0.1
        path: /healthz
        port: probe-port
        scheme: HTTPS
      initialDelaySeconds: 10
      periodSeconds: 10
      timeoutSeconds: 15
    volumeMounts:
    - mountPath: /etc/ssl/certs
      name: ca-certs
      readOnly: true
    - mountPath: /etc/pki/ca-trust
      name: etc-pki-ca-trust
      readOnly: true
    - mountPath: /etc/pki/tls/certs
      name: etc-pki-tls-certs
      readOnly: true
    - mountPath: /usr/libexec/kubernetes/kubelet-plugins/volume/exec
      name: flexvolume-dir
    - mountPath: /etc/kubernetes/pki
      name: k8s-certs
      readOnly: true
    - mountPath: /etc/kubernetes/controller-manager.conf
      name: kubeconfig
      readOnly: true
  hostNetwork: true
  priority: 2000001000
  priorityClassName: system-node-critical
  securityContext:
    seccompProfile:
      type: RuntimeDefault
  volumes:
  - hostPath:
      path: /etc/ssl/certs
      type: DirectoryOrCreate
    name: ca-certs
  - hostPath:
      path: /etc/pki/ca-trust
      type: DirectoryOrCreate
    name: etc-pki-ca-trust
  - hostPath:
      path: /etc/pki/tls/certs
      type: DirectoryOrCreate
    name: etc-pki-tls-certs
  - hostPath:
      path: /usr/libexec/kubernetes/kubelet-plugins/volume/exec
      type: DirectoryOrCreate
    name: flexvolume-dir
  - hostPath:
      path: /etc/kubernetes/pki
      type: DirectoryOrCreate
    name: k8s-certs
  - hostPath:
      path: /etc/kubernetes/controller-manager.conf
      type: FileOrCreate
    name: kubeconfig
status: {}
[root@master-1 storage]# 

[root@master-1 storage]# kubectl get pods -n clickhouse  -o wide -w
NAME           READY   STATUS    RESTARTS   AGE   IP           NODE       NOMINATED NODE   READINESS GATES
clickhouse-0   1/1     Running   0          99s   10.244.6.2   worker-1   <none>           <none>
clickhouse-0   1/1     Running   0          2m19s   10.244.6.2   worker-1   <none>           <none>
clickhouse-0   1/1     Running   0          2m49s   10.244.6.2   worker-1   <none>           <none>
clickhouse-0   1/1     Terminating   0          2m49s   10.244.6.2   worker-1   <none>           <none>
^C[root@master-1 storage]# kubectl get pods -n clickhouse  -o wide -w
NAME           READY   STATUS        RESTARTS   AGE   IP           NODE       NOMINATED NODE   READINESS GATES
clickhouse-0   1/1     Terminating   0          10m   10.244.6.2   worker-1   <none>           <none>

why it still treminate?

how solve this problem